In [15]:
import pandapower as pp
import os
source_path = os.path.join("/home/breveron/data/pylovo_validation/Forchheim/V8/converted_splitted_data/subnets/regular_nets", "LV_028.json")
net = pp.from_json(source_path)

In [16]:
# pp.plotting.geo.convert_geodata_to_geojson(net, delete=True)
# pp.to_json(net, filename="SWF_geo.json")
# pp.to_excel(net, filename="SWF_geo.xlsx")

In [17]:
pp.plotting.geo.dump_to_geojson(net, buses=True, lines=True, switches=True, trafos=True, t_is_3w=True, include_type_id=True)

The supplied network uses an outdated geodata format. Please update your geodata by
running `pandapower.plotting.geo.convert_geodata_to_geojson(net)`


{"features": [], "type": "FeatureCollection"}

In [23]:
pp.plotting.plotly.simple_plotly(net, respect_switches=True, use_line_geo=True, figsize=1.0, aspectratio='auto', line_width=1.0, bus_size=10.0, ext_grid_size=20.0, bus_color='blue', line_color='grey', trafo_color='green', trafo3w_color='green', ext_grid_color='yellow', filename='temp-plot.html', auto_open=False, showlegend=True, additional_traces=None, zoomlevel=11, auto_draw_traces=True, hvdc_color='cyan')

### Transform original xlsx V8 files to geopackage for QGIS  

In [2]:
from pathlib import Path
import colorsys
import json

import geopandas as gpd
import pandas as pd
import pandapower as pp
from shapely.geometry import LineString, shape

from pylovo.analysis.validation_helpers import (
    MINI_GRID_BUS_THRESHOLD,
    _build_lv_topology,
    _assign_buses_to_trafos,
)


# ── Helpers ───────────────────────────────────────────────────────────────────

def _net_number(net_name: str) -> str:
    parts = net_name.split("_", 1)
    return parts[1] if len(parts) > 1 else net_name


def _net_color(net_name: str) -> str:
    if net_name.startswith("MV"):
        return "#5f5f5f"
    number = int(_net_number(net_name))
    hue = ((number * 137) % 360) / 360.0
    red, green, blue = colorsys.hsv_to_rgb(hue, 0.65, 0.9)
    return "#{:02x}{:02x}{:02x}".format(
        int(red * 255), int(green * 255), int(blue * 255)
    )


def _parse_geojson_geometry(value):
    if not isinstance(value, str) or not value.strip():
        return None
    try:
        geojson_obj = json.loads(value)
        if isinstance(geojson_obj, str):
            geojson_obj = json.loads(geojson_obj)
        return shape(geojson_obj)
    except (AttributeError, TypeError, ValueError, json.JSONDecodeError):
        return None


def _extract_lv_subnet_id(chr_name: str) -> str | None:
    """Extract the 3-digit LV subnet ID from a chr_name starting with '7'."""
    if isinstance(chr_name, str) and len(chr_name) > 4 and chr_name.startswith("7"):
        return chr_name[1:4]
    return None


def _read_indexed_sheet(xlsx_path: Path, sheet_name: str) -> pd.DataFrame:
    frame = pd.read_excel(xlsx_path, sheet_name=sheet_name)
    index_column = next((c for c in frame.columns if str(c).startswith("Unnamed: 0")), None)
    if index_column is not None:
        frame = frame.set_index(index_column)
    dup_cols = [c for c in frame.columns if str(c).startswith("Unnamed: 0")]
    if dup_cols:
        frame = frame.drop(columns=dup_cols)
    return frame


def _load_geodata_from_xlsx(xlsx_path: Path, net_index: int):
    """Load bus and line GeoDataFrames from an xlsx subnet file.

    Returns (line_geo, bus_geo).  If the file has no 'line' sheet (e.g.
    mini_grids with 1 bus), line_geo will be an empty GeoDataFrame.
    """
    workbook = pd.ExcelFile(xlsx_path)
    bus_df = _read_indexed_sheet(xlsx_path, "bus")
    bus_geo = gpd.GeoDataFrame(bus_df.copy(), geometry=bus_df["geo"].map(_parse_geojson_geometry))
    bus_geo["net"] = net_index
    bus_geo["plz"] = -1
    bus_geo["consumer_bus"] = bus_geo["name"].astype(str).str.contains("Consumer Nodebus", na=False)

    if "line" not in workbook.sheet_names:
        line_geo = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry")
        line_geo["net"] = net_index
        line_geo["plz"] = -1
        return line_geo, bus_geo

    line_df = _read_indexed_sheet(xlsx_path, "line")

    # Synthesize line geometry from bus endpoints when line.geo is all NaN.
    line_geoms = line_df["geo"].map(_parse_geojson_geometry) if "geo" in line_df.columns else pd.Series(dtype="object")
    if line_geoms.isna().all() and "from_bus" in line_df.columns and "to_bus" in line_df.columns:
        synthesized = []
        for _, row in line_df.iterrows():
            from_pt = bus_geo.geometry.get(row.get("from_bus"))
            to_pt = bus_geo.geometry.get(row.get("to_bus"))
            if from_pt is not None and to_pt is not None and not from_pt.is_empty and not to_pt.is_empty:
                synthesized.append(LineString([from_pt, to_pt]))
            else:
                synthesized.append(None)
        line_geoms = pd.Series(synthesized, index=line_df.index)

    line_geo = gpd.GeoDataFrame(line_df.copy(), geometry=line_geoms)
    line_geo["net"] = net_index
    line_geo["plz"] = -1
    return line_geo, bus_geo


# ── Load original SWF.json (single authoritative source for trafos) ──────────

swf_path = Path("/home/breveron/data/SWF.json")
swf_net = pp.from_json(str(swf_path))

export_dir = Path("qgis_exports")
export_dir.mkdir(parents=True, exist_ok=True)

# Build topology-based subnet bus counts (same logic as extract_lv_grids).
G_lv, lv_bus_set = _build_lv_topology(swf_net)
topo_subnet_buses = _assign_buses_to_trafos(swf_net, G_lv, lv_bus_set)
topo_bus_counts = {name.split("_", 1)[1]: len(buses) for name, buses in topo_subnet_buses.items()}

# ── Build transformer layer from SWF.json net.trafo ──────────────────────────
# Only in-service trafos whose lv_bus has chr_name starting with "7" (= LV
# trafos with vn_kv = 0.4).  Includes both regular and mini_grid trafos.

trafo_df = swf_net.trafo[swf_net.trafo["in_service"] == True].copy()

trafo_rows = []
for trafo_idx, trafo_row in trafo_df.iterrows():
    lv_bus = trafo_row["lv_bus"]
    if lv_bus not in swf_net.bus.index:
        continue
    lv_chr = str(swf_net.bus.at[lv_bus, "chr_name"])
    sub_id = _extract_lv_subnet_id(lv_chr)
    if sub_id is None:
        continue

    geom = _parse_geojson_geometry(swf_net.bus.at[lv_bus, "geo"])
    if geom is None:
        continue

    bus_count = topo_bus_counts.get(sub_id, 0)
    net_name = f"LV_{sub_id}"
    trafo_rows.append({
        "geometry": geom,
        "trafo_index": trafo_idx,
        "trafo_name": trafo_row.get("name", ""),
        "transformer_mva": float(trafo_row["sn_mva"]),
        "transformer_kva": round(float(trafo_row["sn_mva"]) * 1000),
        "net_name": net_name,
        "net_number": sub_id,
        "style_color": _net_color(net_name),
        "feature_type": "transformer",
        "bus_count": bus_count,
        "grid_category": "mini_grid" if bus_count < MINI_GRID_BUS_THRESHOLD else "regular",
        "geometry_source": "SWF.json",
    })

trafo_layer = gpd.GeoDataFrame(trafo_rows, geometry="geometry", crs="EPSG:25832")
n_regular_trafos = (trafo_layer["grid_category"] == "regular").sum()
n_mini_trafos = (trafo_layer["grid_category"] == "mini_grid").sum()

# ── Build bus + line layers from split subnet xlsx files ──────────────────────

base_dir = Path("/home/breveron/data/regular_nets")
subnet_dirs = [base_dir / "regular", base_dir / "mini_grids"]

xlsx_by_name = {}
for d in subnet_dirs:
    if d.exists():
        for p in d.glob("*.xlsx"):
            xlsx_by_name[p.stem] = p

subnet_names = sorted(xlsx_by_name.keys())
if not subnet_names:
    raise FileNotFoundError(f"No subnet exports found under: {base_dir}")

line_frames = []
bus_frames = []

for net_index, net_name in enumerate(subnet_names):
    xlsx_path = xlsx_by_name[net_name]
    net_color = _net_color(net_name)
    net_number = _net_number(net_name)

    line_geo, bus_geo = _load_geodata_from_xlsx(xlsx_path, net_index=net_index)

    line_geo = line_geo.loc[line_geo.geometry.notna()].copy()
    bus_geo = bus_geo.loc[bus_geo.geometry.notna()].copy()

    if line_geo.empty and bus_geo.empty:
        continue

    if not line_geo.empty:
        line_geo = line_geo.set_crs("EPSG:25832", allow_override=True)
    if not bus_geo.empty:
        bus_geo = bus_geo.set_crs("EPSG:25832", allow_override=True)

    line_geo["net_name"] = net_name
    line_geo["net_number"] = net_number
    line_geo["voltage_level"] = "MV" if net_name.startswith("MV") else "LV"
    line_geo["line_role"] = "mv_line" if net_name.startswith("MV") else "lv_line"
    line_geo["style_color"] = net_color
    line_geo["geometry_source"] = "xlsx"

    bus_geo["net_name"] = net_name
    bus_geo["net_number"] = net_number
    bus_geo["voltage_level"] = "MV" if net_name.startswith("MV") else "LV"
    bus_geo["bus_role"] = "lv_bus" if net_name.startswith("LV") else "mv_bus"
    bus_geo["style_color"] = net_color
    bus_geo["geometry_source"] = "xlsx"

    if not line_geo.empty:
        line_frames.append(line_geo.drop(columns=["geo"], errors="ignore"))
    bus_frames.append(bus_geo.drop(columns=["geo"], errors="ignore"))

if not line_frames:
    raise ValueError("No line geometries could be created from the subnet files.")

line_layer = gpd.GeoDataFrame(pd.concat(line_frames, ignore_index=True), geometry="geometry", crs="EPSG:25832")
bus_layer = gpd.GeoDataFrame(pd.concat(bus_frames, ignore_index=True), geometry="geometry", crs="EPSG:25832")

# ── Export ────────────────────────────────────────────────────────────────────

line_export = export_dir / "SWF_lines.geojson"
bus_export = export_dir / "SWF_buses.geojson"
gpkg_export = export_dir / "SWF.gpkg"

line_layer.to_file(line_export, driver="GeoJSON")
bus_layer.to_file(bus_export, driver="GeoJSON")

if gpkg_export.exists():
    gpkg_export.unlink()

line_layer.to_file(gpkg_export, layer="lines", driver="GPKG")
bus_layer.to_file(gpkg_export, layer="buses", driver="GPKG", mode="a")
trafo_layer.to_file(gpkg_export, layer="transformers", driver="GPKG", mode="a")

print(f"Source: {swf_path}")
print(f"Subnet file count: {len(subnet_names)} (regular + mini_grids + MV)")
print(f"Line features: {len(line_layer)}")
print(f"Bus features: {len(bus_layer)}")
print(f"Transformer features: {len(trafo_layer)} (in-service LV only)")
print(f"  Regular grid trafos (>={MINI_GRID_BUS_THRESHOLD} buses): {n_regular_trafos}")
print(f"  Mini grid trafos (<{MINI_GRID_BUS_THRESHOLD} buses): {n_mini_trafos}")
print(f"GeoPackage: {gpkg_export.resolve()}")

Source: /home/breveron/data/SWF.json
Subnet file count: 186 (regular + mini_grids + MV)
Line features: 33990
Bus features: 32840
Transformer features: 185 (in-service LV only)
  Regular grid trafos (>=5 buses): 138
  Mini grid trafos (<5 buses): 47
GeoPackage: /home/breveron/git/github/pylovo/validation/grid_preparation/qgis_exports/SWF.gpkg


In [ ]:
from pylovo.analysis.validation_helpers import export_synthetic_grids_to_excel

# ── Option A: export a single grid ────────────────────────────────────────────
written = export_synthetic_grids_to_excel(
    output_dir="validation/synthetic_grids",
    plz=91301,
    kcid=1,
    bcid=3,
)

# ── Option B: export all grids for the current VERSION_ID ────────────────────
# written = export_synthetic_grids_to_excel(
#     output_dir="validation/synthetic_grids",
# )

print(f"Exported {len(written)} file(s):")
for p in written:
    print(f"  {p}")